In [7]:
import os
import json

def compile_txt_poems_to_jsonl(output_jsonl_path: str):
    """
    Traverses the mounted Kaggle input directory, locates all nested .txt files,
    extracts the raw poem content preserving line and stanza structure, 
    and packages them into a unified pre-training JSONL file.
    """
    base_input_dir = "/kaggle/input"
    
    if not os.path.exists(base_input_dir) or not os.listdir(base_input_dir):
        raise FileNotFoundError("Kaggle input directory is completely empty. Attach the dataset in the right-side panel.")

    # Locate the target dataset root directory dynamically
    attached_roots = []
    for root, dirs, _ in os.walk(base_input_dir):
        if 'poemsdataset' in root:
            attached_roots.append(root)
            break
            
    if not attached_roots:
        # Fallback to the first available directory depth
        for root, dirs, _ in os.walk(base_input_dir):
            if dirs and not root.endswith('datasets'):
                attached_roots.append(os.path.join(root, dirs[0]))
                break
                
    if not attached_roots:
        target_input_dir = base_input_dir
    else:
        target_input_dir = attached_roots[0]

    print(f"Targeting resolved dataset directory: {target_input_dir}")
    
    # Collect all text files recursively
    txt_paths = []
    for root, _, files in os.walk(target_input_dir):
        for file in files:
            if file.lower().endswith('.txt'):
                txt_paths.append(os.path.join(root, file))
                
    total_files = len(txt_paths)
    if total_files == 0:
        raise FileNotFoundError(f"Zero text (.txt) files located under path: {target_input_dir}")

    print(f"Successfully located {total_files} text files. Beginning serialization stream...")

    processed_poems = 0
    skipped_files = 0

    with open(output_jsonl_path, 'w', encoding='utf-8') as out_file:
        for idx, file_path in enumerate(txt_paths):
            try:
                # 'ignore' handles any remaining corrupted multi-byte OS characters safely
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    poem_body = f.read().strip()
                
                if not poem_body or len(poem_body) < 10:
                    skipped_files += 1
                    continue

                # Structure the content with sequence boundary markers for causal model pre-training
                structured_sequence = (
                    f"<|startoftext|>\n"
                    f"<|type|> poem\n"
                    f"<|content|>\n"
                    f"{poem_body}\n"
                    f"<|endoftext|>"
                )

                # Stream directly to storage disk
                record = {"text": structured_sequence}
                out_file.write(json.dumps(record, ensure_ascii=False) + '\n')
                processed_poems += 1

            except Exception:
                skipped_files += 1
                continue

            if (idx + 1) % 1000 == 0 or (idx + 1) == total_files:
                print(f"Progress: Processed {idx + 1}/{total_files} files | Valid Sequences Serialized: {processed_poems}")

    print("\n[COMPLETED] Pre-training Dataset Compilation Successful.")
    print(f"Destination Output: {output_jsonl_path}")
    print(f"Total Poems Processed: {processed_poems}")
    print(f"Total Files Skipped: {skipped_files}")

if __name__ == "__main__":
    OUTPUT_FILE_PATH = "/kaggle/working/poems_pretraining_corpus.jsonl"
    compile_txt_poems_to_jsonl(OUTPUT_FILE_PATH)

Targeting resolved dataset directory: /kaggle/input/datasets/michaelarman/poemsdataset
Successfully located 20657 text files. Beginning serialization stream...
Progress: Processed 1000/20657 files | Valid Sequences Serialized: 1000
Progress: Processed 2000/20657 files | Valid Sequences Serialized: 1999
Progress: Processed 3000/20657 files | Valid Sequences Serialized: 2999
Progress: Processed 4000/20657 files | Valid Sequences Serialized: 3999
Progress: Processed 5000/20657 files | Valid Sequences Serialized: 4999
Progress: Processed 6000/20657 files | Valid Sequences Serialized: 5999
Progress: Processed 7000/20657 files | Valid Sequences Serialized: 6999
Progress: Processed 8000/20657 files | Valid Sequences Serialized: 7999
Progress: Processed 9000/20657 files | Valid Sequences Serialized: 8999
Progress: Processed 10000/20657 files | Valid Sequences Serialized: 9999
Progress: Processed 11000/20657 files | Valid Sequences Serialized: 10999
Progress: Processed 12000/20657 files | Valid

In [8]:
!wc -l /kaggle/working/poems_pretraining_corpus.jsonl
!head -n 1 /kaggle/working/poems_pretraining_corpus.jsonl

20656 /kaggle/working/poems_pretraining_corpus.jsonl
{"text": "<|startoftext|>\n<|type|> poem\n<|content|>\nLay a garland on my hearse,\nOf the dismal yew,\nMaidens, willow branches bear,\nSay I died true.\nMy love was false, but I was firm\nFrom my hour of birth;\nUpon my buried body lie\nLightly, gentle earth.\n<|endoftext|>"}
